In [30]:
import wrds

db = wrds.Connection()

WRDS recommends setting up a .pgpass file.
Created .pgpass file successfully.
You can create this file yourself at any time with the create_pgpass_file() function.
Loading library list...
Done


In [31]:
libs = db.list_libraries()
print(len(libs))
print(sorted(libs))

330
['aha_sample', 'ahasamp', 'audit', 'audit_acct_os', 'audit_audit_comp', 'audit_common', 'auditsmp', 'auditsmp_all', 'bank', 'bank_all', 'bank_premium_samp', 'banksamp', 'block', 'block_all', 'boardex', 'boardex_na', 'boardex_trial', 'boardsmp', 'bvd_amadeus_trial', 'bvd_bvdbankf_trial', 'bvd_orbis_trial', 'bvdsamp', 'calcbench_trial', 'calcbnch', 'candid_samp', 'cboe', 'cboe_all', 'cboe_sample', 'cboesamp', 'cddsamp', 'ciq', 'ciq_common', 'ciqsamp', 'ciqsamp_capstrct', 'ciqsamp_common', 'ciqsamp_keydev', 'ciqsamp_pplintel', 'ciqsamp_ratings', 'ciqsamp_transactions', 'ciqsamp_transcripts', 'cisdmsmp', 'columnar', 'comp', 'comp_execucomp', 'comp_filings', 'comp_global_daily', 'comp_na_daily_all', 'comp_ph', 'comp_pit', 'comp_segments_hist_daily', 'comp_snapshot', 'comp_urq', 'comph', 'compsamp', 'compsamp_all', 'compsamp_computext', 'compsamp_snapshot', 'compseg', 'compsnap', 'contrib', 'contrib_as_filed_financials', 'contrib_bond_dickerson', 'contrib_bond_firm_link', 'contrib_ceo_tu

In [32]:
keywords = ("index", "idx", "sp500", "spx", "comp", "crsp")
candidates = [l for l in sorted(libs) if any(k in l.lower() for k in keywords)]
print(candidates)

['audit_audit_comp', 'comp', 'comp_execucomp', 'comp_filings', 'comp_global_daily', 'comp_na_daily_all', 'comp_ph', 'comp_pit', 'comp_segments_hist_daily', 'comp_snapshot', 'comp_urq', 'comph', 'compsamp', 'compsamp_all', 'compsamp_computext', 'compsamp_snapshot', 'compseg', 'compsnap', 'crsp', 'crsp_a_ccm', 'crsp_a_indexes', 'crsp_a_stock', 'crsp_q_indexhist', 'crsp_q_mutualfunds', 'crspsamp', 'crspsamp_all', 'crspsamp_mf', 'execcomp', 'wrdsapps_link_comp_eushort', 'wrdsapps_link_crsp_bond', 'wrdsapps_link_crsp_comp_bdx', 'wrdsapps_link_crsp_factset', 'wrdsapps_link_crsp_ibes', 'wrdsapps_link_crsp_optionm', 'wrdsapps_link_crsp_taq', 'wrdsapps_link_crsp_taqm']


In [33]:
comp_tables = db.list_tables(library="comp")
idx_tables = [t for t in sorted(comp_tables) if "idx" in t.lower()]
print(len(idx_tables))
print(idx_tables)

16
['g_idx_daily', 'g_idx_index', 'g_idx_mth', 'g_idxcst_his', 'idx_ann', 'idx_anndes', 'idx_daily', 'idx_index', 'idx_mth', 'idx_qrt', 'idx_qrtdes', 'idxcst_his', 'r_idxclscd', 'r_majidxcl', 'spidx_cst', 'wrds_idx_cst_current']


In [34]:
db.describe_table(library="comp", table="idx_index")

Approximately 2257 rows in comp.idx_index.


,name,nullable,type,comment
0,conm,True,VARCHAR(70),Index Name
1,gvkeyx,True,VARCHAR(7),Global Index Key - Index
2,idx13key,True,VARCHAR(14),13 Character Key
3,idxcstflg,True,VARCHAR(3),Index Constituent Flag
4,idxstat,True,VARCHAR(3),Index Active/Inactive Flag
5,indexcat,True,VARCHAR(11),Index Category Code
6,indexgeo,True,VARCHAR(11),Index Geographical Area
7,indexid,True,VARCHAR(11),Index Identifier
8,indextype,True,VARCHAR(11),Index Code Type
9,indexval,True,VARCHAR(11),Index Code Value


In [35]:
idx_master = db.get_table(library="comp", table="idx_index", obs=-1)
print(idx_master.shape)
print(idx_master.columns.tolist())

(2257, 14)
['conm', 'gvkeyx', 'idx13key', 'idxcstflg', 'idxstat', 'indexcat', 'indexgeo', 'indexid', 'indextype', 'indexval', 'spii', 'spmi', 'tic', 'tici']


In [36]:
cols = idx_master.columns.tolist()
name_col = "conm" if "conm" in cols else [c for c in cols if "nam" in c.lower()][0]

mask = idx_master[name_col].astype(str).str.contains(
    "sector|GICS|S&P 500", case=False, na=False
)
hits = idx_master[mask]
print(hits.shape)

show_cols = [c for c in ["gvkeyx", "conm", "indexcat", "indextype", "tic", "idxstkcd"] if c in cols]
print(hits[show_cols].to_string())

(20, 14)
      gvkeyx                                                                    conm   indexcat   indextype     tic
244   131001                                            SP400 Multi-Sector Hldgs .SI        S&P     GSUBIND  DFI.M3
365   131002                                            SP500 Multi-Sector Hldgs .SI        S&P     GSUBIND  DFI.I3
630   000003                                                        S&P 500 Comp-Ltd        S&P       LGCAP   I0003
633   026113                                              S&P 500/Barra Growth Index  S&P/BARRA      GROWTH   I0017
636   026114                                               S&P 500/Barra Value Index  S&P/BARRA       VALUE   I0018
645   000002  S&P 500 Ex-Financials, Real Estate, Utilities and Transportation Index        S&P   COMPOSITE   I0002
824   131000                                            SP1500 Multi-Sector Hldg .SI        S&P     GSUBIND  DFI.X3
945   131018                                            SP600 M

In [37]:
sector_kw = "Energy|Financ|Technolog|Health|Industrial|Material|Utilit|Staples|Discretion|Communicat|Real Estate"
mask = idx_master["conm"].astype(str).str.contains(sector_kw, case=False, na=False)
hits = idx_master[mask]
print(hits.shape)
print(hits[["gvkeyx", "conm", "indexcat", "indextype", "tic"]].to_string())

(354, 14)
      gvkeyx                                                                    conm indexcat   indextype     tic
0     131796                                               SP1500 Water Utilities .I      S&P        GIND   WET.X
9     132338                                            SP1500 Electric Utilities .I      S&P        GIND   ELC.X
10    132342                                                 SP1500 Gas Utilities .I      S&P        GIND   GAS.X
11    132346                                               SP1500 Multi-Utilities .I      S&P        GIND   MUL.X
12    130061                                              SP600 Energy Equip&Svcs .I      S&P        GIND   EES.S
15    130074                                             SP600 Construct Material .I      S&P        GIND   CMT.S
43    131398                                             SP600 Health Care Prvdrs .I      S&P        GIND   HPS.S
44    131430                                                  SP600 Biotechnol

In [38]:
gsector = idx_master[idx_master["indextype"] == "GSECTOR"].copy()
sp500_sectors = gsector[gsector["tic"].astype(str).str.endswith(".I")]
print(sp500_sectors.shape)
print(sp500_sectors[["gvkeyx", "conm", "tic"]].to_string())

(11, 14)
      gvkeyx                          conm    tic
678   129001     SP500 Consumer Staples .S  CST.I
679   128940   SP500 Consumr Discretion .S  CDI.I
680   128798               SP500 Energy .S  ENE.I
681   128859            SP500 Materials .S  MAT.I
682   128898          SP500 Industrials .S  IND.I
683   129059     SP500 Information Tech .S  TEC.I
684   129021          SP500 Health Care .S  HEA.I
685   129039           SP500 Financials .S  FIN.I
686   129081  SP500 COMMUNICATION SERVICES  TEL.I
687   129218            SP500 Utilities .S  UTL.I
1680  027228           SP500 REAL ESTATE.S  REE.I


In [39]:
db.describe_table(library="comp", table="idx_daily")

Approximately 8645699 rows in comp.idx_daily.


,name,nullable,type,comment
0,gvkeyx,True,VARCHAR(7),Global Index Key - Index Daily
1,dvpsxd,True,"NUMERIC(28, 8)",Index Daily Dividends
2,newnum,True,INTEGER,Index Number - New
3,oldnum,True,INTEGER,Index Number - Old
4,prccd,True,"NUMERIC(28, 8)",Index Price - Close Daily
5,prccddiv,True,"NUMERIC(28, 8)",Index Value - Total Return
6,prccddivn,True,"NUMERIC(28, 8)",Index Value - Total Return - Net Dividends
7,prchd,True,"NUMERIC(28, 8)",Index Price - High Daily
8,prcld,True,"NUMERIC(28, 8)",Index Price - Low Daily
9,datadate,True,DATE,Data Date - Index Daily


In [40]:
gvkeyx_list = [
    "128798", "128859", "128898", "128940", "129001",
    "129059", "129021", "129039", "129081", "129218", "027228",
]
gvkeyx_str = ", ".join(f"'{g}'" for g in gvkeyx_list)

query = f"""
    SELECT datadate, gvkeyx, prccddiv, prccddivn, prccd, dvpsxd
    FROM comp.idx_daily
    WHERE gvkeyx IN ({gvkeyx_str})
    ORDER BY gvkeyx, datadate
"""
raw = db.raw_sql(query, date_cols=["datadate"])
print(raw.shape)
print(raw.head())
print(raw.tail())

(95050, 6)
    datadate  gvkeyx    prccddiv prccddivn       prccd    dvpsxd
0 2016-09-19  027228  370.072348      <NA>  198.126352       0.0
1 2016-09-20  027228  369.458387      <NA>  197.773371  0.024284
2 2016-09-21  027228   373.93221      <NA>  200.168236       0.0
3 2016-09-22  027228  381.173862      <NA>  203.980259  0.064479
4 2016-09-23  027228  382.255119      <NA>  204.558879       0.0
        datadate  gvkeyx     prccddiv prccddivn       prccd    dvpsxd
95045 2026-06-11  129218  1489.360781      <NA>  444.959075       0.0
95046 2026-06-12  129218   1506.05342      <NA>  449.946142       0.0
95047 2026-06-15  129218  1513.444742      <NA>  452.039456  0.114905
95048 2026-06-16  129218  1523.830252      <NA>  455.141426       0.0
95049 2026-06-17  129218  1504.109436      <NA>  449.251164       0.0


In [41]:
name_map = {
    "128798": "Energy", "128859": "Materials", "128898": "Industrials",
    "128940": "ConsDiscretionary", "129001": "ConsStaples",
    "129059": "InfoTech", "129021": "HealthCare", "129039": "Financials",
    "129081": "CommServices", "129218": "Utilities", "027228": "RealEstate",
}
cov = (raw.groupby("gvkeyx")
          .agg(start=("datadate", "min"),
               end=("datadate", "max"),
               n=("datadate", "size"),
               n_nan_tr=("prccddiv", lambda s: s.isna().sum()))
          .reset_index())
cov["sector"] = cov["gvkeyx"].map(name_map)
print(cov.to_string())

    gvkeyx      start        end     n  n_nan_tr             sector
0   027228 2016-09-19 2026-06-17  2450       0.0         RealEstate
1   128798 1989-09-11 2026-06-17  9260       0.0             Energy
2   128859 1989-09-11 2026-06-17  9260       0.0          Materials
3   128898 1989-09-11 2026-06-17  9260       0.0        Industrials
4   128940 1989-09-11 2026-06-17  9260       0.0  ConsDiscretionary
5   129001 1989-09-11 2026-06-17  9260       0.0        ConsStaples
6   129021 1989-09-11 2026-06-17  9260       0.0         HealthCare
7   129039 1989-09-11 2026-06-17  9260       0.0         Financials
8   129059 1989-09-11 2026-06-17  9260       0.0           InfoTech
9   129081 1989-09-11 2026-06-17  9260       0.0       CommServices
10  129218 1989-09-11 2026-06-17  9260       0.0          Utilities


In [42]:
import numpy as np
import pandas as pd

name_map = {
    "128798": "Energy", "128859": "Materials", "128898": "Industrials",
    "128940": "ConsDiscretionary", "129001": "ConsStaples",
    "129059": "InfoTech", "129021": "HealthCare", "129039": "Financials",
    "129081": "CommServices", "129218": "Utilities", "027228": "RealEstate",
}

df = raw.copy()
df["sector"] = df["gvkeyx"].map(name_map)
df["datadate"] = pd.to_datetime(df["datadate"])
df["prccddiv"] = pd.to_numeric(df["prccddiv"], errors="coerce")
df["prccd"] = pd.to_numeric(df["prccd"], errors="coerce")

print("=" * 60)
print("[1] DUPLICATE (sector, date) ROWS")
print("=" * 60)
dup = df.duplicated(subset=["gvkeyx", "datadate"], keep=False)
print("dup row count:", int(dup.sum()))
if dup.sum():
    print(df[dup].sort_values(["gvkeyx", "datadate"]).to_string())

print("\n" + "=" * 60)
print("[2] NON-MONOTONIC / UNSORTED DATES PER SECTOR")
print("=" * 60)
for g, sub in df.groupby("gvkeyx"):
    d = sub["datadate"].values
    if not (d[:-1] <= d[1:]).all():
        print(name_map[g], "NOT sorted ascending")
print("check done")

print("\n" + "=" * 60)
print("[3] TOTAL-RETURN LEVEL: NON-POSITIVE OR NULL")
print("=" * 60)
bad_lvl = df[(df["prccddiv"].isna()) | (df["prccddiv"] <= 0)]
print("bad level rows:", len(bad_lvl))
if len(bad_lvl):
    print(bad_lvl[["sector", "datadate", "prccddiv"]].to_string())

print("\n" + "=" * 60)
print("[4] DAILY RETURN OUTLIERS (|ret| > 25%) FROM prccddiv")
print("=" * 60)
df = df.sort_values(["gvkeyx", "datadate"])
df["ret"] = df.groupby("gvkeyx")["prccddiv"].pct_change()
out = df[df["ret"].abs() > 0.25]
print("outlier rows:", len(out))
if len(out):
    print(out[["sector", "datadate", "prccddiv", "ret"]].to_string())
print("\nret describe per sector:")
print(df.groupby("sector")["ret"].describe()[["min", "max", "std"]].to_string())

print("\n" + "=" * 60)
print("[5] EXACT-ZERO RETURNS (possible stale/carry-forward)")
print("=" * 60)
zero = df[df["ret"] == 0.0]
print("zero-ret rows:", len(zero))
print(zero.groupby("sector").size().to_string() if len(zero) else "none")

print("\n" + "=" * 60)
print("[6] CALENDAR GAPS > 5 DAYS BETWEEN CONSECUTIVE OBS")
print("=" * 60)
df["gap"] = df.groupby("gvkeyx")["datadate"].diff().dt.days
big = df[df["gap"] > 5]
print("gap>5 rows:", len(big))
if len(big):
    print(big[["sector", "datadate", "gap"]].sort_values("gap", ascending=False).head(30).to_string())

print("\n" + "=" * 60)
print("[7] WEEKEND-DATED OBSERVATIONS (dow >= 5)")
print("=" * 60)
wknd = df[df["datadate"].dt.dayofweek >= 5]
print("weekend rows:", len(wknd))
if len(wknd):
    print(wknd.groupby(wknd["datadate"].dt.dayofweek).size().to_string())

print("\n" + "=" * 60)
print("[8] DATE-AXIS ALIGNMENT ACROSS SECTORS")
print("=" * 60)
piv = df.pivot_table(index="datadate", columns="sector", values="prccddiv", aggfunc="first")
print("union calendar length:", piv.shape[0])
print("rows with ALL 11 present:", int(piv.notna().all(axis=1).sum()))
print("rows with ANY missing:", int(piv.isna().any(axis=1).sum()))
print("\nfirst date where all 11 present:",
      piv.index[piv.notna().all(axis=1)].min())
print("per-sector missing count on union calendar:")
print(piv.isna().sum().to_string())

print("\n" + "=" * 60)
print("[9] TR vs PRICE SANITY (cumulative TR should >= price drift)")
print("=" * 60)
for g, sub in df.groupby("gvkeyx"):
    tr = sub["prccddiv"].iloc[-1] / sub["prccddiv"].iloc[0] - 1
    pr = sub["prccd"].iloc[-1] / sub["prccd"].iloc[0] - 1
    flag = "" if tr >= pr else "  <-- TR < PR ??"
    print(f"{name_map[g]:18s} TR={tr:7.2%}  PR={pr:7.2%}{flag}")

print("\n" + "=" * 60)
print("[10] COMM SERVICES 2018 RECLASS DISCONTINUITY CHECK")
print("=" * 60)
cs = df[df["gvkeyx"] == "129081"].copy()
for yr in [2018, 2019]:
    around = cs[(cs["datadate"] >= f"{yr}-09-15") & (cs["datadate"] <= f"{yr}-10-05")]
    print(f"\n{yr} late-Sep window (ret tail):")
    print(around[["datadate", "prccddiv", "ret"]].to_string())

[1] DUPLICATE (sector, date) ROWS
dup row count: 0

[2] NON-MONOTONIC / UNSORTED DATES PER SECTOR
check done

[3] TOTAL-RETURN LEVEL: NON-POSITIVE OR NULL
bad level rows: 0

[4] DAILY RETURN OUTLIERS (|ret| > 25%) FROM prccddiv
outlier rows: 0

ret describe per sector:
                        min       max       std
sector                                         
CommServices      -0.104408  0.137947  0.013473
ConsDiscretionary -0.120744  0.131057  0.013297
ConsStaples       -0.092213  0.092385  0.009498
Energy            -0.199939  0.184844  0.016032
Financials        -0.170056  0.187736  0.016704
HealthCare        -0.099925  0.124253    0.0115
Industrials       -0.114446   0.12752  0.012443
InfoTech          -0.139121  0.174413  0.016702
Materials         -0.121313  0.132841  0.013756
RealEstate        -0.165108  0.087959  0.013041
Utilities         -0.115427  0.135251   0.01124

[5] EXACT-ZERO RETURNS (possible stale/carry-forward)
zero-ret rows: 0
none

[6] CALENDAR GAPS > 5 DAYS B

In [43]:
START = "2016-09-19"

panel = (df[df["datadate"] >= START]
         .pivot_table(index="datadate", columns="sector",
                      values="prccddiv", aggfunc="first")
         .sort_index())

SECTORS = ["Energy", "Materials", "Industrials", "ConsDiscretionary",
           "ConsStaples", "InfoTech", "HealthCare", "Financials",
           "CommServices", "Utilities", "RealEstate"]
panel = panel[SECTORS]

print("panel shape:", panel.shape)
print("date range:", panel.index.min().date(), "->", panel.index.max().date())
print("any NaN in level panel:", int(panel.isna().sum().sum()))
print("per-sector NaN:", panel.isna().sum().to_dict())

rets = np.log(panel).diff().dropna(how="any")
print("\nlog-return panel shape:", rets.shape)
print("any NaN in return panel:", int(rets.isna().sum().sum()))

idx = panel.index
print("\nstrictly increasing dates:", bool((idx[1:] > idx[:-1]).all()))
print("duplicate dates:", int(idx.duplicated().sum()))

print("\nall-present rows on common calendar:",
      int(panel.notna().all(axis=1).sum()), "/", panel.shape[0])

panel shape: (2450, 11)
date range: 2016-09-19 -> 2026-06-17
any NaN in level panel: 0
per-sector NaN: {'Energy': 0, 'Materials': 0, 'Industrials': 0, 'ConsDiscretionary': 0, 'ConsStaples': 0, 'InfoTech': 0, 'HealthCare': 0, 'Financials': 0, 'CommServices': 0, 'Utilities': 0, 'RealEstate': 0}

log-return panel shape: (2449, 11)
any NaN in return panel: 0

strictly increasing dates: True
duplicate dates: 0

all-present rows on common calendar: 2450 / 2450


In [44]:
weekly_ret = rets.resample("W-FRI").sum()

obs_per_week = rets.resample("W-FRI").size()
print("weeks total:", len(weekly_ret))
print("trading-days-per-week distribution:")
print(obs_per_week.value_counts().sort_index().to_string())

empty_weeks = int((obs_per_week == 0).sum())
print("empty weeks (0 trading days):", empty_weeks)

weekly_ret = weekly_ret[obs_per_week > 0]
print("weeks after dropping empty:", len(weekly_ret))

state = weekly_ret.idxmin(axis=1)
state.name = "laggard"
print("\nstate sequence length:", len(state))
print("\nlaggard frequency by sector:")
print(state.value_counts().reindex(SECTORS).to_string())

print("\nfirst 20 states:")
print(state.head(20).to_string())

weeks total: 509
trading-days-per-week distribution:
3      1
4     94
5    414
empty weeks (0 trading days): 0
weeks after dropping empty: 509

state sequence length: 509

laggard frequency by sector:
laggard
Energy               127
Materials             27
Industrials           13
ConsDiscretionary     40
ConsStaples           34
InfoTech              43
HealthCare            34
Financials            34
CommServices          53
Utilities             61
RealEstate            43

first 20 states:
datadate
2016-09-23               Energy
2016-09-30            Utilities
2016-10-07           RealEstate
2016-10-14           HealthCare
2016-10-21         CommServices
2016-10-28           RealEstate
2016-11-04             InfoTech
2016-11-11            Utilities
2016-11-18           HealthCare
2016-11-25           HealthCare
2016-12-02             InfoTech
2016-12-09           HealthCare
2016-12-16          Industrials
2016-12-23    ConsDiscretionary
2016-12-30             InfoTech
2017-01-

In [45]:
seq = state.tolist()
S = SECTORS
idx_of = {s: i for i, s in enumerate(S)}
k = len(S)

N = np.zeros((k, k), dtype=int)
for a, b in zip(seq[:-1], seq[1:]):
    N[idx_of[a], idx_of[b]] += 1

row_tot = N.sum(axis=1)
P = np.divide(N, row_tot[:, None], where=row_tot[:, None] > 0,
              out=np.zeros((k, k)))

N_df = pd.DataFrame(N, index=S, columns=S)
P_df = pd.DataFrame(P, index=S, columns=S)

print("total transitions:", N.sum(), "(expected", len(seq) - 1, ")")
print("\nrow totals (n_i, sample per state):")
print(pd.Series(row_tot, index=S).to_string())

print("\nzero-count cells:", int((N == 0).sum()), "/", k * k)
print("zero rows (states never seen as origin):",
      [S[i] for i in range(k) if row_tot[i] == 0])

print("\nrow-sum check (should be 1.0 or 0):")
print(pd.Series(P.sum(axis=1), index=S).round(6).to_string())

print("\nTransition count matrix N:")
print(N_df.to_string())

total transitions: 508 (expected 508 )

row totals (n_i, sample per state):
Energy               126
Materials             27
Industrials           13
ConsDiscretionary     40
ConsStaples           34
InfoTech              43
HealthCare            34
Financials            34
CommServices          53
Utilities             61
RealEstate            43

zero-count cells: 7 / 121
zero rows (states never seen as origin): []

row-sum check (should be 1.0 or 0):
Energy               1.0
Materials            1.0
Industrials          1.0
ConsDiscretionary    1.0
ConsStaples          1.0
InfoTech             1.0
HealthCare           1.0
Financials           1.0
CommServices         1.0
Utilities            1.0
RealEstate           1.0

Transition count matrix N:
                   Energy  Materials  Industrials  ConsDiscretionary  ConsStaples  InfoTech  HealthCare  Financials  CommServices  Utilities  RealEstate
Energy                 29          7            2                 13           14    

In [46]:
n_states = len(SECTORS)
free_params = n_states * (n_states - 1)
print("n states:", n_states)
print("free parameters in P:", free_params, "= 11 x 10")

n_daily = len(rets)
n_weekly = len(state)
print("\nobserved transitions:")
print("  daily :", n_daily - 1)
print("  weekly:", n_weekly - 1)
print("\ntransitions per free parameter:")
print("  daily : {:.2f}".format((n_daily - 1) / free_params))
print("  weekly: {:.2f}".format((n_weekly - 1) / free_params))

alpha = 1.0
N_smooth = N + alpha
P_smooth = N_smooth / N_smooth.sum(axis=1, keepdims=True)

P_smooth_df = pd.DataFrame(P_smooth, index=SECTORS, columns=SECTORS)

print("\nzero cells before smoothing:", int((N == 0).sum()))
print("zero cells after smoothing :", int((P_smooth == 0).sum()))
print("row sums after smoothing:", np.allclose(P_smooth.sum(axis=1), 1.0))

print("\nmax abs change vs raw P (per cell):",
      round(np.nanmax(np.abs(P_smooth - P)), 4))
print("rows most affected (L1 shift):")
l1 = pd.Series(np.abs(P_smooth - P).sum(axis=1), index=SECTORS)
print(l1.sort_values(ascending=False).round(3).to_string())

n states: 11
free parameters in P: 110 = 11 x 10

observed transitions:
  daily : 2448
  weekly: 508

transitions per free parameter:
  daily : 22.25
  weekly: 4.62

zero cells before smoothing: 7
zero cells after smoothing : 0
row sums after smoothing: True

max abs change vs raw P (per cell): 0.0702
rows most affected (L1 shift):
Industrials          0.288
Materials            0.187
Financials           0.167
ConsStaples          0.124
HealthCare           0.123
InfoTech             0.117
ConsDiscretionary    0.113
RealEstate           0.100
CommServices         0.094
Utilities            0.077
Energy               0.038


In [47]:
def stationary_distribution(P_mat, states):
    vals, vecs = np.linalg.eig(P_mat.T)
    i = np.argmin(np.abs(vals - 1.0))
    pi = np.real(vecs[:, i])
    pi = pi / pi.sum()
    return pd.Series(pi, index=states), vals[i]

pi_raw, ev_raw = stationary_distribution(P, SECTORS)
pi_smooth, ev_smooth = stationary_distribution(P_smooth, SECTORS)

print("eigenvalue used (should be ~1):")
print("  raw    :", np.round(ev_raw, 6))
print("  smooth :", np.round(ev_smooth, 6))

print("\nstationary pi (raw vs smoothed):")
comp = pd.DataFrame({"raw": pi_raw, "smooth": pi_smooth}).round(4)
comp["empirical_freq"] = (state.value_counts().reindex(SECTORS) / len(state)).round(4)
print(comp.to_string())

print("\nsanity:")
print("  raw pi sums to   :", round(pi_raw.sum(), 6))
print("  smooth pi sums to:", round(pi_smooth.sum(), 6))
print("  raw any negative :", bool((pi_raw < 0).any()))
print("  smooth any neg   :", bool((pi_smooth < 0).any()))

print("\n  pi_smooth . P_smooth - pi_smooth (max abs residual):",
      round(np.max(np.abs(pi_smooth.values @ P_smooth - pi_smooth.values)), 8))

eigenvalue used (should be ~1):
  raw    : (1+0j)
  smooth : (1+0j)

stationary pi (raw vs smoothed):
                      raw  smooth  empirical_freq
Energy             0.2480  0.2178          0.2495
Materials          0.0531  0.0604          0.0530
Industrials        0.0256  0.0382          0.0255
ConsDiscretionary  0.0787  0.0811          0.0786
ConsStaples        0.0669  0.0715          0.0668
InfoTech           0.0846  0.0859          0.0845
HealthCare         0.0669  0.0715          0.0668
Financials         0.0669  0.0715          0.0668
CommServices       0.1043  0.1017          0.1041
Utilities          0.1201  0.1145          0.1198
RealEstate         0.0846  0.0859          0.0845

sanity:
  raw pi sums to   : 1.0
  smooth pi sums to: 1.0
  raw any negative : False
  smooth any neg   : False

  pi_smooth . P_smooth - pi_smooth (max abs residual): 0.0


In [48]:
print("=" * 60)
print("[A] STATE SEQUENCE vs weekly_ret CONSISTENCY")
print("=" * 60)
recomputed_state = weekly_ret.idxmin(axis=1)
print("state matches recomputed idxmin:", bool((state == recomputed_state).all()))
print("state index == weekly_ret index:", bool(state.index.equals(weekly_ret.index)))
print("state length:", len(state), "| weekly_ret rows:", len(weekly_ret))

print("\n" + "=" * 60)
print("[B] TIE CHECK IN idxmin (silent arbitrary picks)")
print("=" * 60)
row_min = weekly_ret.min(axis=1)
tie_counts = (weekly_ret.eq(row_min, axis=0)).sum(axis=1)
print("weeks with >1 sector at the min:", int((tie_counts > 1).sum()))
if (tie_counts > 1).any():
    print(weekly_ret[tie_counts > 1])

print("\n" + "=" * 60)
print("[C] N MATRIX REBUILT INDEPENDENTLY")
print("=" * 60)
seq2 = state.tolist()
N2 = np.zeros((len(SECTORS), len(SECTORS)), dtype=int)
pos = {s: i for i, s in enumerate(SECTORS)}
for a, b in zip(seq2[:-1], seq2[1:]):
    N2[pos[a], pos[b]] += 1
print("N matches independent rebuild:", bool((N == N2).all()))
print("N total == len(seq)-1:", int(N.sum()) == len(seq2) - 1)
print("N row sums == origin counts (excl last state):")
origin_counts = pd.Series(seq2[:-1]).value_counts().reindex(SECTORS).fillna(0).astype(int)
print("  match:", bool((N.sum(axis=1) == origin_counts.values).all()))

print("\n" + "=" * 60)
print("[D] P_smooth TRACEABILITY (is it really N+alpha?)")
print("=" * 60)
P_check = (N + 1.0) / (N + 1.0).sum(axis=1, keepdims=True)
print("P_smooth == recomputed (N+1) normalized:", np.allclose(P_smooth, P_check))
print("P (raw) == N normalized:",
      np.allclose(P, np.divide(N, N.sum(axis=1, keepdims=True),
                  where=N.sum(axis=1, keepdims=True) > 0,
                  out=np.zeros_like(P)), equal_nan=True))

print("\n" + "=" * 60)
print("[E] STATIONARY pi CROSS-CHECK (eigvec vs power iteration)")
print("=" * 60)
v = np.ones(len(SECTORS)) / len(SECTORS)
for _ in range(100000):
    v_new = v @ P_smooth
    if np.max(np.abs(v_new - v)) < 1e-14:
        break
    v = v_new
print("power-iteration vs eig pi (max abs diff):",
      round(float(np.max(np.abs(v - pi_smooth.values))), 12))

print("\n" + "=" * 60)
print("[F] LOG-RETURN AGGREGATION SANITY")
print("=" * 60)
wk = rets.resample("W-FRI").sum()
wk = wk[rets.resample("W-FRI").size() > 0]
recon = np.exp(wk.sum(axis=0)) - 1
direct = panel.iloc[-1] / panel.iloc[0] - 1
print("cum return via weekly-logsum vs panel endpoints (max abs diff):")
print("  ", round(float((recon - direct).abs().max()), 8))

print("\n" + "=" * 60)
print("[G] COLUMN/INDEX ORDER CONSISTENCY ACROSS OBJECTS")
print("=" * 60)
print("panel cols == SECTORS:", list(panel.columns) == SECTORS)
print("rets cols == SECTORS:", list(rets.columns) == SECTORS)
print("weekly_ret cols == SECTORS:", list(weekly_ret.columns) == SECTORS)
print("P_smooth_df index == SECTORS:", list(P_smooth_df.index) == SECTORS)
print("P_smooth_df cols == SECTORS:", list(P_smooth_df.columns) == SECTORS)

print("\n" + "=" * 60)
print("[H] RESAMPLE BOUNDARY LEAK CHECK")
print("=" * 60)
print("daily rows used:", len(rets))
print("sum of per-week trading-day counts:",
      int(rets.resample("W-FRI").size().sum()))
print("  match (no rows lost/double-counted):",
      len(rets) == int(rets.resample("W-FRI").size().sum()))

[A] STATE SEQUENCE vs weekly_ret CONSISTENCY
state matches recomputed idxmin: True
state index == weekly_ret index: True
state length: 509 | weekly_ret rows: 509

[B] TIE CHECK IN idxmin (silent arbitrary picks)
weeks with >1 sector at the min: 0

[C] N MATRIX REBUILT INDEPENDENTLY
N matches independent rebuild: True
N total == len(seq)-1: True
N row sums == origin counts (excl last state):
  match: True

[D] P_smooth TRACEABILITY (is it really N+alpha?)
P_smooth == recomputed (N+1) normalized: True
P (raw) == N normalized: True

[E] STATIONARY pi CROSS-CHECK (eigvec vs power iteration)
power-iteration vs eig pi (max abs diff): 0.0

[F] LOG-RETURN AGGREGATION SANITY
cum return via weekly-logsum vs panel endpoints (max abs diff):
   0.0

[G] COLUMN/INDEX ORDER CONSISTENCY ACROSS OBJECTS
panel cols == SECTORS: True
rets cols == SECTORS: True
weekly_ret cols == SECTORS: True
P_smooth_df index == SECTORS: True
P_smooth_df cols == SECTORS: True

[H] RESAMPLE BOUNDARY LEAK CHECK
daily rows u

In [49]:
P_use = P_smooth
states = SECTORS
k = len(states)

recurrence = 1.0 / pi_smooth
print("mean recurrence time (1/pi), weeks:")
print(recurrence.round(2).sort_values().to_string())

def hitting_times_to(target_idx, Pm):
    n = Pm.shape[0]
    others = [i for i in range(n) if i != target_idx]
    Q = Pm[np.ix_(others, others)]
    A = np.eye(len(others)) - Q
    b = np.ones(len(others))
    h_other = np.linalg.solve(A, b)
    h = np.zeros(n)
    for pos_i, i in enumerate(others):
        h[i] = h_other[pos_i]
    return h

H = np.zeros((k, k))
for j in range(k):
    H[:, j] = hitting_times_to(j, P_use)

H_df = pd.DataFrame(H, index=states, columns=states)
print("\nhitting time H[i,j] = mean weeks from i to first reach j:")
print(H_df.round(1).to_string())

print("\nsanity checks:")
print("  diagonal of H (should be 0):", np.round(np.diag(H), 8))
print("  any negative hitting time:", bool((H < 0).any()))
print("  recurrence vs return-via-hitting (Kac) check:")
ret_via_hit = np.array([1 + P_use[j, :] @ (H[:, j]) for j in range(k)])
kac = pd.DataFrame({"1/pi": recurrence.values,
                    "1+sum_k P_jk h_kj": ret_via_hit},
                   index=states).round(3)
print(kac.to_string())
print("  max abs diff (Kac identity):",
      round(float(np.max(np.abs(recurrence.values - ret_via_hit))), 8))

mean recurrence time (1/pi), weeks:
Energy                4.59
Utilities             8.74
CommServices          9.83
InfoTech             11.65
RealEstate           11.65
ConsDiscretionary    12.33
ConsStaples          13.98
HealthCare           13.98
Financials           13.98
Materials            16.55
Industrials          26.21

hitting time H[i,j] = mean weeks from i to first reach j:
                   Energy  Materials  Industrials  ConsDiscretionary  ConsStaples  InfoTech  HealthCare  Financials  CommServices  Utilities  RealEstate
Energy                0.0       16.4         26.8               11.8         13.4      11.7        14.9        15.1          10.2        8.0        12.3
Materials             4.4        0.0         26.7               12.1         14.1      11.5        14.5        14.1          10.8        8.4        12.0
Industrials           5.0       16.7          0.0               12.1         14.4      11.3        13.8        14.3           9.8        8.7        1

In [50]:
def eig_summary(Pm, label):
    vals, vecs = np.linalg.eig(Pm)
    order = np.argsort(-np.abs(vals))
    vals = vals[order]
    vecs = vecs[:, order]
    mod = np.abs(vals)
    print(f"--- {label} ---")
    tab = pd.DataFrame({
        "Re": np.round(vals.real, 5),
        "Im": np.round(vals.imag, 5),
        "modulus": np.round(mod, 5),
        "is_complex": np.abs(vals.imag) > 1e-9,
    })
    print(tab.to_string())
    print("sum of eigenvalues (=trace):", round(vals.sum().real, 5),
          "| trace(P):", round(np.trace(Pm), 5))
    print("largest eigenvalue:", np.round(vals[0], 8))
    print("number of complex eigenvalues:", int((np.abs(vals.imag) > 1e-9).sum()))
    return vals, vecs, mod

vals_raw, vecs_raw, mod_raw = eig_summary(P, "RAW P-hat")
print()
vals_s, vecs_s, mod_s = eig_summary(P_smooth, "SMOOTHED P-hat")

print("\n" + "=" * 50)
print("CROSS-CHECKS (smoothed)")
print("=" * 50)
print("largest modulus == 1 (Perron):", round(mod_s[0], 8))
print("all moduli <= 1:", bool((mod_s <= 1 + 1e-9).all()))
print("second-largest modulus |lambda_2|:", round(mod_s[1], 5))
print("spectral gap g = 1 - |lambda_2|:", round(1 - mod_s[1], 5))
print("complex pairs come conjugate:",
      np.allclose(sorted(vals_s.imag), sorted(-vals_s.imag)))

--- RAW P-hat ---
         Re       Im  modulus  is_complex
0   1.00000  0.00000  1.00000       False
1   0.11761  0.01288  0.11831        True
2   0.11761 -0.01288  0.11831        True
3   0.01250  0.10250  0.10326        True
4   0.01250 -0.10250  0.10326        True
5  -0.07579  0.05295  0.09245        True
6  -0.07579 -0.05295  0.09245        True
7  -0.09158  0.00000  0.09158       False
8   0.01385  0.03756  0.04003        True
9   0.01385 -0.03756  0.04003        True
10 -0.00327  0.00000  0.00327       False
sum of eigenvalues (=trace): 1.04152 | trace(P): 1.04152
largest eigenvalue: (1+0j)
number of complex eigenvalues: 8

--- SMOOTHED P-hat ---
         Re       Im  modulus  is_complex
0   1.00000  0.00000  1.00000       False
1   0.09572  0.01311  0.09661        True
2   0.09572 -0.01311  0.09661        True
3  -0.07696  0.00000  0.07696       False
4   0.00446  0.07666  0.07679        True
5   0.00446 -0.07666  0.07679        True
6  -0.04636  0.02174  0.05121        True
7

In [51]:
rng = np.random.default_rng(42)

def lambda2_from_sequence(seq_list, states, alpha=1.0):
    pos = {s: i for i, s in enumerate(states)}
    m = len(states)
    Nm = np.zeros((m, m))
    for a, b in zip(seq_list[:-1], seq_list[1:]):
        Nm[pos[a], pos[b]] += 1
    Pm = (Nm + alpha) / (Nm + alpha).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm)
    ev = ev[np.argsort(-np.abs(ev))]
    return np.abs(ev[1])

actual_seq = state.tolist()
actual_l2 = lambda2_from_sequence(actual_seq, SECTORS)
print("actual |lambda_2| (smoothed):", round(actual_l2, 5))

print("\n" + "=" * 55)
print("[1] PERMUTATION TEST: shuffle destroys time-structure")
print("=" * 55)
B = 2000
null_l2 = np.empty(B)
arr = np.array(actual_seq, dtype=object)
for b in range(B):
    perm = rng.permutation(arr)
    null_l2[b] = lambda2_from_sequence(list(perm), SECTORS)

p_val = (np.sum(null_l2 >= actual_l2) + 1) / (B + 1)
print(f"null |lambda_2| over {B} shuffles:")
print("  mean :", round(null_l2.mean(), 5))
print("  std  :", round(null_l2.std(), 5))
print("  2.5% / 50% / 97.5%:",
      np.round(np.percentile(null_l2, [2.5, 50, 97.5]), 5))
print("  max  :", round(null_l2.max(), 5))
print(f"\nactual = {actual_l2:.5f}  | p-value (actual >= null) = {p_val:.4f}")
print("interpretation:",
      "INSIDE noise band -> no detectable memory" if actual_l2 <= np.percentile(null_l2, 97.5)
      else "OUTSIDE noise band -> real structure")

print("\n" + "=" * 55)
print("[2] POSITIVE CONTROL: inject a cycle, can pipeline catch it?")
print("=" * 55)
cycle = ["Energy", "Utilities", "InfoTech", "Financials"]
p_stay = 0.15
T_syn = 509
syn = [cycle[0]]
for _ in range(T_syn - 1):
    cur = syn[-1]
    if cur in cycle and rng.random() > p_stay:
        nxt = cycle[(cycle.index(cur) + 1) % len(cycle)]
    else:
        nxt = rng.choice(SECTORS)
    syn.append(nxt)

pos = {s: i for i, s in enumerate(SECTORS)}
Nsyn = np.zeros((11, 11))
for a, b in zip(syn[:-1], syn[1:]):
    Nsyn[pos[a], pos[b]] += 1
Psyn = (Nsyn + 1.0) / (Nsyn + 1.0).sum(axis=1, keepdims=True)
ev_syn = np.linalg.eigvals(Psyn)
ev_syn = ev_syn[np.argsort(-np.abs(ev_syn))]

print("injected cycle:", " -> ".join(cycle), "-> (repeat)")
print("synthetic |lambda_2|:", round(abs(ev_syn[1]), 5))
print("synthetic top-4 eigenvalues:")
for v in ev_syn[:4]:
    print(f"   Re={v.real:+.4f}  Im={v.imag:+.4f}  mod={abs(v):.4f}")
print("\nlargest imag part among eigenvalues:",
      round(np.max(np.abs(ev_syn.imag)), 5),
      "(should be clearly > 0 if cycle is detected)")
print("compare: real-data largest imag part:",
      round(np.max(np.abs(np.linalg.eigvals(P_smooth).imag)), 5))

actual |lambda_2| (smoothed): 0.09661

[1] PERMUTATION TEST: shuffle destroys time-structure
null |lambda_2| over 2000 shuffles:
  mean : 0.1165
  std  : 0.01884
  2.5% / 50% / 97.5%: [0.08657 0.11439 0.16086]
  max  : 0.20688

actual = 0.09661  | p-value (actual >= null) = 0.8766
interpretation: INSIDE noise band -> no detectable memory

[2] POSITIVE CONTROL: inject a cycle, can pipeline catch it?
injected cycle: Energy -> Utilities -> InfoTech -> Financials -> (repeat)
synthetic |lambda_2|: 0.76397
synthetic top-4 eigenvalues:
   Re=+1.0000  Im=+0.0000  mod=1.0000
   Re=-0.7640  Im=+0.0000  mod=0.7640
   Re=+0.0056  Im=+0.7479  mod=0.7479
   Re=+0.0056  Im=-0.7479  mod=0.7479

largest imag part among eigenvalues: 0.74785 (should be clearly > 0 if cycle is detected)
compare: real-data largest imag part: 0.07666


In [52]:
print("=" * 60)
print("[3] STATE-DEFINITION ROBUSTNESS: laggard vs leader")
print("=" * 60)

leader_state = weekly_ret.idxmax(axis=1)

def full_diag(seq_series, label):
    seq_list = seq_series.tolist()
    pos = {s: i for i, s in enumerate(SECTORS)}
    m = len(SECTORS)
    Nm = np.zeros((m, m))
    for a, b in zip(seq_list[:-1], seq_list[1:]):
        Nm[pos[a], pos[b]] += 1
    Pm = (Nm + 1.0) / (Nm + 1.0).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm)
    ev = ev[np.argsort(-np.abs(ev))]
    l2 = abs(ev[1])

    arr = np.array(seq_list, dtype=object)
    nb = 2000
    nullv = np.empty(nb)
    for b in range(nb):
        perm = rng.permutation(arr)
        Nb = np.zeros((m, m))
        pl = list(perm)
        for a, c in zip(pl[:-1], pl[1:]):
            Nb[pos[a], pos[c]] += 1
        Pb = (Nb + 1.0) / (Nb + 1.0).sum(axis=1, keepdims=True)
        evb = np.linalg.eigvals(Pb)
        nullv[b] = np.abs(evb[np.argsort(-np.abs(evb))][1])
    pv = (np.sum(nullv >= l2) + 1) / (nb + 1)

    print(f"\n--- {label} ---")
    print(f"  |lambda_2|      : {l2:.5f}")
    print(f"  null mean/97.5% : {nullv.mean():.5f} / {np.percentile(nullv,97.5):.5f}")
    print(f"  max imag part   : {np.max(np.abs(ev.imag)):.5f}")
    print(f"  p-value         : {pv:.4f}")
    print(f"  verdict         : {'NOISE (no memory)' if l2 <= np.percentile(nullv,97.5) else 'SIGNAL'}")
    return Nm

full_diag(state, "LAGGARD (argmin) [our main]")
N_leader = full_diag(leader_state, "LEADER (argmax)")

print("\n" + "=" * 60)
print("[4] CHI-SQUARE TEST OF FIRST-ORDER DEPENDENCE")
print("=" * 60)
print("H0: next state independent of current state")

from scipy.stats import chi2_contingency, chi2

for lbl, seqs in [("LAGGARD", state.tolist()), ("LEADER", leader_state.tolist())]:
    pos = {s: i for i, s in enumerate(SECTORS)}
    m = len(SECTORS)
    Nm = np.zeros((m, m))
    for a, b in zip(seqs[:-1], seqs[1:]):
        Nm[pos[a], pos[b]] += 1

    keep = Nm.sum(axis=1) > 0
    Nuse = Nm[keep][:, Nm.sum(axis=0) > 0]
    chi2_stat, p, dof, exp = chi2_contingency(Nuse)
    low_exp = (exp < 5).mean()

    print(f"\n--- {lbl} ---")
    print(f"  chi2 statistic : {chi2_stat:.2f}")
    print(f"  dof            : {dof}")
    print(f"  p-value        : {p:.4f}")
    print(f"  cells exp<5    : {low_exp:.1%}  (high -> chi2 unreliable, small sample)")
    print(f"  verdict        : {'DEPENDENT' if p < 0.05 else 'INDEPENDENT (no structure)'}")

[3] STATE-DEFINITION ROBUSTNESS: laggard vs leader

--- LAGGARD (argmin) [our main] ---
  |lambda_2|      : 0.09661
  null mean/97.5% : 0.11669 / 0.15865
  max imag part   : 0.07666
  p-value         : 0.8756
  verdict         : NOISE (no memory)

--- LEADER (argmax) ---
  |lambda_2|      : 0.16044
  null mean/97.5% : 0.11688 / 0.15931
  max imag part   : 0.07542
  p-value         : 0.0225
  verdict         : SIGNAL

[4] CHI-SQUARE TEST OF FIRST-ORDER DEPENDENCE
H0: next state independent of current state

--- LAGGARD ---
  chi2 statistic : 80.29
  dof            : 100
  p-value        : 0.9265
  cells exp<5    : 77.7%  (high -> chi2 unreliable, small sample)
  verdict        : INDEPENDENT (no structure)

--- LEADER ---
  chi2 statistic : 85.70
  dof            : 100
  p-value        : 0.8452
  cells exp<5    : 71.9%  (high -> chi2 unreliable, small sample)
  verdict        : INDEPENDENT (no structure)


In [53]:
print("=" * 60)
print("[5] DAILY-FREQUENCY RE-TEST (5x sample, lower noise floor)")
print("=" * 60)

daily_lag = rets.idxmin(axis=1)
daily_led = rets.idxmax(axis=1)
print("daily transitions:", len(daily_lag) - 1)

def lambda2_and_spectrum(seq_list):
    pos = {s: i for i, s in enumerate(SECTORS)}; m = len(SECTORS)
    Nm = np.zeros((m, m))
    for a, b in zip(seq_list[:-1], seq_list[1:]):
        Nm[pos[a], pos[b]] += 1
    Pm = (Nm + 1.0) / (Nm + 1.0).sum(axis=1, keepdims=True)
    ev = np.linalg.eigvals(Pm); ev = ev[np.argsort(-np.abs(ev))]
    return abs(ev[1]), ev, Nm

def perm_pvalue(seq_list, nb=2000):
    l2, _, _ = lambda2_and_spectrum(seq_list)
    arr = np.array(seq_list, dtype=object); pos = {s: i for i, s in enumerate(SECTORS)}; m = len(SECTORS)
    nullv = np.empty(nb)
    for b in range(nb):
        pl = list(rng.permutation(arr))
        Nb = np.zeros((m, m))
        for a, c in zip(pl[:-1], pl[1:]): Nb[pos[a], pos[c]] += 1
        Pb = (Nb + 1.0) / (Nb + 1.0).sum(axis=1, keepdims=True)
        evb = np.linalg.eigvals(Pb)
        nullv[b] = abs(evb[np.argsort(-np.abs(evb))][1])
    return l2, nullv, (np.sum(nullv >= l2) + 1) / (nb + 1)

for lbl, sq in [("DAILY laggard", daily_lag.tolist()), ("DAILY leader", daily_led.tolist())]:
    l2, nullv, pv = perm_pvalue(sq)
    _, ev, _ = lambda2_and_spectrum(sq)
    print(f"\n--- {lbl} ---")
    print(f"  |lambda_2|={l2:.5f}  null97.5={np.percentile(nullv,97.5):.5f}  p={pv:.4f}")
    print(f"  max imag part={np.max(np.abs(ev.imag)):.5f}")
    print(f"  lambda_2 is complex: {abs(ev[1].imag) > 1e-6}  (Im={ev[1].imag:+.5f})")
    print(f"  verdict: {'SIGNAL' if l2 > np.percentile(nullv,97.5) else 'NOISE'}")

print("\n" + "=" * 60)
print("[6] WEEKLY LEADER lambda_2: real or complex? (cycle vs persistence)")
print("=" * 60)
_, ev_led_w, _ = lambda2_and_spectrum(leader_state.tolist())
print("weekly leader top-4 eigenvalues:")
for v in ev_led_w[:4]:
    print(f"   Re={v.real:+.5f}  Im={v.imag:+.5f}  mod={abs(v):.5f}",
          "<- lambda_2" if abs(abs(v)-abs(ev_led_w[1]))<1e-9 else "")
print("diagonal mass (self-persistence) weekly leader:")
pos = {s: i for i, s in enumerate(SECTORS)}; Nl = np.zeros((11,11))
for a,b in zip(leader_state.tolist()[:-1], leader_state.tolist()[1:]): Nl[pos[a],pos[b]]+=1
Pl = (Nl+1)/(Nl+1).sum(axis=1,keepdims=True)
print("  mean diagonal P[i,i]:", round(np.mean(np.diag(Pl)),4),
      "| uniform baseline 1/11 =", round(1/11,4))

print("\n" + "=" * 60)
print("[7] BLOCK BOOTSTRAP null (preserves short-run autocorr)")
print("=" * 60)
def block_perm_pvalue(seq_list, blocklen=4, nb=2000):
    l2, _, _ = lambda2_and_spectrum(seq_list)
    arr = np.array(seq_list, dtype=object); n = len(arr)
    pos = {s: i for i, s in enumerate(SECTORS)}; m = len(SECTORS)
    nblocks = int(np.ceil(n / blocklen))
    blocks = [arr[i*blocklen:(i+1)*blocklen] for i in range(nblocks)]
    nullv = np.empty(nb)
    for b in range(nb):
        order = rng.permutation(len(blocks))
        pl = list(np.concatenate([blocks[i] for i in order]))[:n]
        Nb = np.zeros((m, m))
        for a, c in zip(pl[:-1], pl[1:]): Nb[pos[a], pos[c]] += 1
        Pb = (Nb + 1.0) / (Nb + 1.0).sum(axis=1, keepdims=True)
        evb = np.linalg.eigvals(Pb)
        nullv[b] = abs(evb[np.argsort(-np.abs(evb))][1])
    return l2, np.percentile(nullv,97.5), (np.sum(nullv>=l2)+1)/(nb+1)

for lbl, sq in [("weekly laggard", state.tolist()), ("weekly leader", leader_state.tolist())]:
    l2, c975, pv = block_perm_pvalue(sq)
    print(f"  {lbl:16s} |l2|={l2:.5f}  block-null97.5={c975:.5f}  p={pv:.4f}  "
          f"{'SIGNAL' if l2>c975 else 'NOISE'}")

[5] DAILY-FREQUENCY RE-TEST (5x sample, lower noise floor)
daily transitions: 2448

--- DAILY laggard ---
  |lambda_2|=0.05646  null97.5=0.08661  p=0.7681
  max imag part=0.03815
  lambda_2 is complex: True  (Im=+0.02049)
  verdict: NOISE

--- DAILY leader ---
  |lambda_2|=0.05469  null97.5=0.08778  p=0.8446
  max imag part=0.03750
  lambda_2 is complex: True  (Im=+0.03750)
  verdict: NOISE

[6] WEEKLY LEADER lambda_2: real or complex? (cycle vs persistence)
weekly leader top-4 eigenvalues:
   Re=+1.00000  Im=+0.00000  mod=1.00000 
   Re=+0.16044  Im=+0.00000  mod=0.16044 <- lambda_2
   Re=+0.02901  Im=+0.07542  mod=0.08080 
   Re=+0.02901  Im=-0.07542  mod=0.08080 
diagonal mass (self-persistence) weekly leader:
  mean diagonal P[i,i]: 0.1073 | uniform baseline 1/11 = 0.0909

[7] BLOCK BOOTSTRAP null (preserves short-run autocorr)
  weekly laggard   |l2|=0.09661  block-null97.5=0.12599  p=0.6887  NOISE
  weekly leader    |l2|=0.16044  block-null97.5=0.17754  p=0.1179  NOISE


In [54]:
from scipy.stats import pearsonr

def leadlag_matrix(R, label, n_perm=2000):
    Rf = R.astype("float64")
    X = Rf.iloc[:-1].values
    Y = Rf.iloc[1:].values
    cols = list(Rf.columns)
    m = len(cols)
    T = X.shape[0]

    C = np.zeros((m, m))
    Pmat = np.zeros((m, m))
    for i in range(m):
        for j in range(m):
            r, p = pearsonr(X[:, i], Y[:, j])
            C[i, j] = r
            Pmat[i, j] = p

    print("=" * 60)
    print(f"{label}  (lead i -> lag j),  T={T} pairs")
    print("=" * 60)
    print("corr stats: max=%.4f  min=%.4f  mean|c|=%.4f"
          % (C.max(), C.min(), np.abs(C).mean()))

    abs_t = 1.96 / np.sqrt(T)
    print("naive |r| threshold (p<0.05, ~1.96/sqrt(T)):", round(abs_t, 4))
    sig = (Pmat < 0.05)
    print("cells significant at p<0.05 (uncorrected):", int(sig.sum()),
          "/", m * m, "  (expected by chance: %.1f)" % (0.05 * m * m))

    bonf = 0.05 / (m * m)
    sig_b = (Pmat < bonf)
    print("cells significant after Bonferroni (p<%.5f):" % bonf, int(sig_b.sum()))
    if sig_b.sum() > 0:
        for i in range(m):
            for j in range(m):
                if sig_b[i, j]:
                    print("   %-18s -> %-18s  r=%+.4f  p=%.2e"
                          % (cols[i], cols[j], C[i, j], Pmat[i, j]))

    fro = np.sqrt((C ** 2).sum())
    null_fro = np.empty(n_perm)
    for b in range(n_perm):
        perm = rng.permutation(T)
        Yp = Y[perm]
        Xc = X - X.mean(0); Yc = Yp - Yp.mean(0)
        denom = np.sqrt((Xc**2).sum(0))[:, None] * np.sqrt((Yc**2).sum(0))[None, :]
        Cb = (Xc.T @ Yc) / denom
        null_fro[b] = np.sqrt((Cb ** 2).sum())
    p_global = (np.sum(null_fro >= fro) + 1) / (n_perm + 1)
    print("global test: ||C||_F=%.4f  null mean=%.4f  null97.5=%.4f  p=%.4f"
          % (fro, null_fro.mean(), np.percentile(null_fro, 97.5), p_global))
    print("verdict:", "STRUCTURE" if p_global < 0.05 else "NO STRUCTURE (consistent with noise)")
    return C, Pmat

C_w, P_w = leadlag_matrix(weekly_ret, "WEEKLY lead-lag")
print()
C_d, P_d = leadlag_matrix(rets, "DAILY lead-lag")

WEEKLY lead-lag  (lead i -> lag j),  T=508 pairs
corr stats: max=0.0790  min=-0.1516  mean|c|=0.0647
naive |r| threshold (p<0.05, ~1.96/sqrt(T)): 0.087
cells significant at p<0.05 (uncorrected): 27 / 121   (expected by chance: 6.1)
cells significant after Bonferroni (p<0.00041): 0
global test: ||C||_F=0.7952  null mean=0.4696  null97.5=0.8151  p=0.0300
verdict: STRUCTURE

DAILY lead-lag  (lead i -> lag j),  T=2448 pairs
corr stats: max=-0.0392  min=-0.1605  mean|c|=0.0894
naive |r| threshold (p<0.05, ~1.96/sqrt(T)): 0.0396
cells significant at p<0.05 (uncorrected): 120 / 121   (expected by chance: 6.1)
cells significant after Bonferroni (p<0.00041): 94
   Energy             -> ConsDiscretionary   r=-0.0877  p=1.39e-05
   Energy             -> ConsStaples         r=-0.0795  p=8.21e-05
   Energy             -> InfoTech            r=-0.1291  p=1.43e-10
   Energy             -> HealthCare          r=-0.0751  p=1.99e-04
   Energy             -> Financials          r=-0.0737  p=2.64e-04
   E

In [55]:
print("=" * 64)
print("[V1] AUTOCORRELATION CHECK: is mean-reversion the culprit?")
print("=" * 64)
Rf_d = rets.astype("float64")
for lbl, R in [("DAILY", Rf_d), ("WEEKLY", weekly_ret.astype("float64"))]:
    ac = []
    for c in R.columns:
        x = R[c].values
        ac.append(np.corrcoef(x[:-1], x[1:])[0, 1])
    ac = np.array(ac)
    print(f"{lbl}: own lag-1 autocorr  mean={ac.mean():+.4f}  "
          f"min={ac.min():+.4f}  max={ac.max():+.4f}  all_negative={bool((ac<0).all())}")

print("\n" + "=" * 64)
print("[V2] MARKET-DEMEANED lead-lag (remove common factor)")
print("=" * 64)

def leadlag_demeaned(R, label, n_perm=2000):
    Rf = R.astype("float64")
    M = Rf.mean(axis=1)
    D = Rf.sub(M, axis=0)
    X = D.iloc[:-1].values
    Y = D.iloc[1:].values
    cols = list(Rf.columns); m = len(cols); T = X.shape[0]

    Xc = X - X.mean(0); Yc = Y - Y.mean(0)
    denom = np.sqrt((Xc**2).sum(0))[:, None] * np.sqrt((Yc**2).sum(0))[None, :]
    C = (Xc.T @ Yc) / denom
    fro = np.sqrt((C**2).sum())

    null_fro = np.empty(n_perm)
    for b in range(n_perm):
        sh = rng.integers(1, T)
        Yp = np.roll(Y, sh, axis=0)
        Ypc = Yp - Yp.mean(0)
        Cb = (Xc.T @ Ypc) / (np.sqrt((Xc**2).sum(0))[:, None] *
                             np.sqrt((Ypc**2).sum(0))[None, :])
        null_fro[b] = np.sqrt((Cb**2).sum())
    p = (np.sum(null_fro >= fro) + 1) / (n_perm + 1)

    off = C[~np.eye(m, dtype=bool)]
    print(f"\n--- {label} (market-demeaned) T={T} ---")
    print(f"  off-diag corr: max={off.max():+.4f} min={off.min():+.4f} "
          f"mean|c|={np.abs(off).mean():.4f}  frac_negative={(off<0).mean():.2f}")
    print(f"  ||C||_F={fro:.4f}  cyclic-shift null mean={null_fro.mean():.4f} "
          f"97.5={np.percentile(null_fro,97.5):.4f}  p={p:.4f}")
    print(f"  verdict: {'STRUCTURE survives' if p<0.05 else 'GONE (was common-factor artifact)'}")
    return C

C_d_dm = leadlag_demeaned(rets, "DAILY")
C_w_dm = leadlag_demeaned(weekly_ret, "WEEKLY")

print("\n" + "=" * 64)
print("[V3] PLACEBO LAG: correlation at large lag (should be ~0)")
print("=" * 64)
def placebo(R, lag, label):
    Rf = R.astype("float64")
    M = Rf.mean(axis=1); D = Rf.sub(M, axis=0).values
    X = D[:-lag]; Y = D[lag:]
    Xc = X - X.mean(0); Yc = Y - Y.mean(0)
    C = (Xc.T @ Yc) / (np.sqrt((Xc**2).sum(0))[:,None]*np.sqrt((Yc**2).sum(0))[None,:])
    print(f"  {label} lag={lag}: ||C||_F={np.sqrt((C**2).sum()):.4f}")

placebo(rets, 1, "DAILY")
placebo(rets, 5, "DAILY")
placebo(rets, 20, "DAILY")

[V1] AUTOCORRELATION CHECK: is mean-reversion the culprit?
DAILY: own lag-1 autocorr  mean=-0.0853  min=-0.1386  max=-0.0392  all_negative=True
WEEKLY: own lag-1 autocorr  mean=-0.0706  min=-0.1465  max=+0.0790  all_negative=False

[V2] MARKET-DEMEANED lead-lag (remove common factor)

--- DAILY (market-demeaned) T=2448 ---
  off-diag corr: max=+0.0884 min=-0.0616 mean|c|=0.0241  frac_negative=0.51
  ||C||_F=0.3221  cyclic-shift null mean=0.2228 97.5=0.2748  p=0.0040
  verdict: STRUCTURE survives

--- WEEKLY (market-demeaned) T=508 ---
  off-diag corr: max=+0.1445 min=-0.1284 mean|c|=0.0376  frac_negative=0.45
  ||C||_F=0.5608  cyclic-shift null mean=0.4831 97.5=0.5816  p=0.0705
  verdict: GONE (was common-factor artifact)

[V3] PLACEBO LAG: correlation at large lag (should be ~0)
  DAILY lag=1: ||C||_F=0.3221
  DAILY lag=5: ||C||_F=0.1961
  DAILY lag=20: ||C||_F=0.2371


In [56]:
print("=" * 64)
print("[W4-1a] CIRCULAR-LAW DISK BENCHMARK for stochastic matrix")
print("=" * 64)
n = len(SECTORS)

ev_s = np.linalg.eigvals(P_smooth)
ev_s = ev_s[np.argsort(-np.abs(ev_s))]
l2_mod = abs(ev_s[1])

row_var = []
for i in range(n):
    p_i = P_smooth[i]
    row_var.append(np.sum(p_i * (1 - p_i)))
sigma2 = np.mean(row_var) / n
R_theory = np.sqrt(sigma2 * n) / np.sqrt(N.sum(axis=1).mean())

print("n states:", n)
print("naive disk radius ~ 1/sqrt(n):", round(1/np.sqrt(n), 4))
nbar = N.sum(axis=1).mean()
R_simple = 1.0 / np.sqrt(nbar)
print("avg transitions per origin row (n_i bar):", round(nbar, 1))
print("noise-scaled radius ~ 1/sqrt(n_i_bar):", round(R_simple, 4))
print("|lambda_2| observed:", round(l2_mod, 4))
print("verdict:",
      "INSIDE noise disk -> noise" if l2_mod <= R_simple
      else "OUTSIDE -> potential signal")

print("\nall non-Perron eigenvalue moduli vs disk:")
for k, v in enumerate(ev_s):
    if k == 0:
        print(f"  lambda_1 = {v.real:+.4f}{v.imag:+.4f}i  mod={abs(v):.4f} (Perron)")
    else:
        inside = "inside" if abs(v) <= R_simple else "OUTSIDE"
        print(f"  lambda_{k+1} mod={abs(v):.4f}  [{inside}]")

print("\n" + "=" * 64)
print("[W4-1b] IRREDUCIBILITY & APERIODICITY")
print("=" * 64)

def is_irreducible(Pmat, tol=1e-12):
    n = Pmat.shape[0]
    reach = (Pmat > tol).astype(int)
    R = reach.copy()
    M = reach.copy()
    for _ in range(n):
        M = (M @ reach > 0).astype(int)
        R = ((R + M) > 0).astype(int)
    return bool((R == 1).all()), R

irr_raw, _ = is_irreducible(P)
irr_s, _ = is_irreducible(P_smooth)
print("raw P irreducible    :", irr_raw)
print("smoothed P irreducible:", irr_s)

diag_pos_raw = int((np.diag(P) > 0).sum())
diag_pos_s = int((np.diag(P_smooth) > 0).sum())
print("\nstates with self-loop P[i,i] > 0 (raw):", diag_pos_raw, "/", n)
print("states with self-loop P[i,i] > 0 (smooth):", diag_pos_s, "/", n)
print("=> any self-loop in irreducible chain implies APERIODIC (period 1)")

def period_gcd(Pmat, tol=1e-12):
    n = Pmat.shape[0]
    returns = []
    Pk = np.eye(n)
    for k in range(1, 3 * n + 1):
        Pk = Pk @ Pmat
        if Pk[0, 0] > tol:
            returns.append(k)
    from math import gcd
    g = 0
    for r in returns:
        g = gcd(g, r)
    return g

print("\nperiod of state 0 (gcd of return times), smoothed:", period_gcd(P_smooth))
print("aperiodic (period==1):", period_gcd(P_smooth) == 1)
print("\nconsequence: irreducible + aperiodic => unique stationary pi,")
print("convergence from any start (ergodic). matches our pi result.")

[W4-1a] CIRCULAR-LAW DISK BENCHMARK for stochastic matrix
n states: 11
naive disk radius ~ 1/sqrt(n): 0.3015
avg transitions per origin row (n_i bar): 46.2
noise-scaled radius ~ 1/sqrt(n_i_bar): 0.1472
|lambda_2| observed: 0.0966
verdict: INSIDE noise disk -> noise

all non-Perron eigenvalue moduli vs disk:
  lambda_1 = +1.0000+0.0000i  mod=1.0000 (Perron)
  lambda_2 mod=0.0966  [inside]
  lambda_3 mod=0.0966  [inside]
  lambda_4 mod=0.0770  [inside]
  lambda_5 mod=0.0768  [inside]
  lambda_6 mod=0.0768  [inside]
  lambda_7 mod=0.0512  [inside]
  lambda_8 mod=0.0512  [inside]
  lambda_9 mod=0.0467  [inside]
  lambda_10 mod=0.0467  [inside]
  lambda_11 mod=0.0094  [inside]

[W4-1b] IRREDUCIBILITY & APERIODICITY
raw P irreducible    : True
smoothed P irreducible: True

states with self-loop P[i,i] > 0 (raw): 10 / 11
states with self-loop P[i,i] > 0 (smooth): 11 / 11
=> any self-loop in irreducible chain implies APERIODIC (period 1)

period of state 0 (gcd of return times), smoothed: 1
ap

In [57]:
from scipy.stats import chi2 as chi2dist

seq = state.tolist()
pos = {s: i for i, s in enumerate(SECTORS)}
m = len(SECTORS)

print("=" * 64)
print("[W4-2a] MARKOV ORDER TEST: order-0 vs order-1 (LR)")
print("=" * 64)
N1 = np.zeros((m, m))
for a, b in zip(seq[:-1], seq[1:]):
    N1[pos[a], pos[b]] += 1
row = N1.sum(axis=1)
col = N1.sum(axis=0)
tot = N1.sum()

LR0 = 0.0
for i in range(m):
    for j in range(m):
        if N1[i, j] > 0:
            p_ij = N1[i, j] / row[i]
            p_j = col[j] / tot
            LR0 += 2 * N1[i, j] * np.log(p_ij / p_j)
df0 = (m - 1) * (m - 1)
p0 = 1 - chi2dist.cdf(LR0, df0)
print(f"H0: order-0 (i.i.d.)  vs  H1: order-1 (Markov)")
print(f"LR stat={LR0:.2f}  df={df0}  p={p0:.4f}")
print(f"verdict: {'order-1 needed (memory)' if p0 < 0.05 else 'order-0 sufficient (i.i.d.)'}")
print(f"AIC: order0={2*0 - 2*loglik_order0 if False else 'see below'}")

def loglik(counts):
    r = counts.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P_ = np.where(r > 0, counts / r, 0)
        ll = np.sum(np.where(counts > 0, counts * np.log(np.where(P_ > 0, P_, 1)), 0))
    return ll

ll1 = loglik(N1)
p_marg = col / tot
ll0 = np.sum([N1[i, j] * np.log(p_marg[j]) for i in range(m) for j in range(m) if N1[i, j] > 0 and p_marg[j] > 0])
k0 = m - 1
k1 = m * (m - 1)
print(f"\nloglik order0={ll0:.2f} (k={k0})  | order1={ll1:.2f} (k={k1})")
print(f"AIC  order0={-2*ll0+2*k0:.1f}  order1={-2*ll1+2*k1:.1f}  "
      f"-> prefer {'order1' if (-2*ll1+2*k1)<(-2*ll0+2*k0) else 'order0'}")
print(f"BIC  order0={-2*ll0+k0*np.log(tot):.1f}  order1={-2*ll1+k1*np.log(tot):.1f}  "
      f"-> prefer {'order1' if (-2*ll1+k1*np.log(tot))<(-2*ll0+k0*np.log(tot)) else 'order0'}")

print("\n" + "=" * 64)
print("[W4-2b] ORDER-1 vs ORDER-2 (feasibility under sparsity)")
print("=" * 64)
pairs = list(zip(seq[:-2], seq[1:-1], seq[2:]))
N2_dict = {}
for a, b, c in pairs:
    N2_dict.setdefault((a, b), np.zeros(m))
    N2_dict[(a, b)][pos[c]] += 1
observed_pairs = len(N2_dict)
print(f"order-2 needs P over {m*m}=121 predecessor-pairs x {m} = {m*m*m} params")
print(f"distinct (prev,cur) pairs actually observed: {observed_pairs} / {m*m}")
print(f"total order-2 transitions available: {len(pairs)}")
print(f"avg transitions per observed pair: {len(pairs)/observed_pairs:.2f}")
print("=> with ~%.1f obs per pair, order-2 MLE is degenerate (most rows 0 or 1 count)" % (len(pairs)/observed_pairs))
single = sum(1 for k, v in N2_dict.items() if v.sum() == 1)
print(f"pairs seen exactly once (zero df): {single} / {observed_pairs}")
print("verdict: order-2 NOT estimable with this sample; cannot reject order-1 from above")

print("\n" + "=" * 64)
print("[W4-2c] TIME-HOMOGENEITY: calm vs crisis (split-sample LR)")
print("=" * 64)
covid = pd.Timestamp("2020-02-20")
covid_end = pd.Timestamp("2020-06-30")
crisis_mask = (state.index >= covid) & (state.index <= covid_end)
print(f"crisis window: {covid.date()} to {covid_end.date()}  weeks={int(crisis_mask.sum())}")

def counts_from(sub_seq):
    Nx = np.zeros((m, m))
    for a, b in zip(sub_seq[:-1], sub_seq[1:]):
        Nx[pos[a], pos[b]] += 1
    return Nx

half = len(seq) // 2
Na = counts_from(seq[:half])
Nb = counts_from(seq[half:])
Npool = Na + Nb

def ll_from(Nx):
    r = Nx.sum(axis=1, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        P_ = np.where(r > 0, Nx / r, 0)
        return np.sum(np.where(Nx > 0, Nx * np.log(np.where(P_ > 0, P_, 1)), 0))

LR_h = 2 * (ll_from(Na) + ll_from(Nb) - ll_from(Npool))
nonzero_pool = int((Npool > 0).sum())
df_h = nonzero_pool - m
p_h = 1 - chi2dist.cdf(LR_h, max(df_h, 1))
print(f"first-half vs second-half split (median date {state.index[half].date()}):")
print(f"  LR={LR_h:.2f}  approx df={df_h}  p={p_h:.4f}")
print(f"  verdict: {'NON-homogeneous (P changes over time)' if p_h < 0.05 else 'homogeneous (cannot reject stability)'}")
low = (Npool < 5).mean()
print(f"  cells with pooled count <5: {low:.1%}  -> chi2 approx {'UNRELIABLE' if low>0.2 else 'ok'}")
print("\nnote: 2016-start sample has only COVID-2020 as a crisis; no 2008.")
print("time-homogeneity power is limited; result is suggestive, not definitive.")

[W4-2a] MARKOV ORDER TEST: order-0 vs order-1 (LR)
H0: order-0 (i.i.d.)  vs  H1: order-1 (Markov)
LR stat=87.68  df=100  p=0.8057
verdict: order-0 sufficient (i.i.d.)
AIC: order0=see below

loglik order0=-1141.49 (k=10)  | order1=-1097.65 (k=110)
AIC  order0=2303.0  order1=2415.3  -> prefer order0
BIC  order0=2345.3  order1=2880.6  -> prefer order0

[W4-2b] ORDER-1 vs ORDER-2 (feasibility under sparsity)
order-2 needs P over 121=121 predecessor-pairs x 11 = 1331 params
distinct (prev,cur) pairs actually observed: 114 / 121
total order-2 transitions available: 507
avg transitions per observed pair: 4.45
=> with ~4.4 obs per pair, order-2 MLE is degenerate (most rows 0 or 1 count)
pairs seen exactly once (zero df): 17 / 114
verdict: order-2 NOT estimable with this sample; cannot reject order-1 from above

[W4-2c] TIME-HOMOGENEITY: calm vs crisis (split-sample LR)
crisis window: 2020-02-20 to 2020-06-30  weeks=19
first-half vs second-half split (median date 2021-08-06):
  LR=128.47  appro

In [59]:
GVKEYX_MAP = name_map
ALPHA = 1.0

SECTORS10 = [s for s in SECTORS if s != "RealEstate"]
GVKEYX10 = {g: n for g, n in GVKEYX_MAP.items() if n != "RealEstate"}

pos10 = {s: i for i, s in enumerate(SECTORS10)}
M10 = len(SECTORS10)

df_lh = raw.copy()
df_lh["sector"] = df_lh["gvkeyx"].map(GVKEYX_MAP)
df_lh = df_lh[df_lh["sector"].isin(SECTORS10)].copy()
df_lh["datadate"] = pd.to_datetime(df_lh["datadate"])
df_lh["prccddiv"] = pd.to_numeric(df_lh["prccddiv"], errors="coerce")

panel_lh = (df_lh.pivot_table(index="datadate", columns="sector",
                              values="prccddiv", aggfunc="first")
            .sort_index()[SECTORS10].astype("float64"))
rets_lh = np.log(panel_lh).diff().dropna(how="any")

print("long-history panel:", panel_lh.shape,
      "|", panel_lh.index.min().date(), "->", panel_lh.index.max().date())

weekly_lh = rets_lh.resample("W-FRI").sum()
weekly_lh = weekly_lh[rets_lh.resample("W-FRI").size() > 0]
state_lh = weekly_lh.idxmin(axis=1)
seq_lh = state_lh.tolist()
print("weekly states:", len(seq_lh))

N_lh = np.zeros((M10, M10))
for a, b in zip(seq_lh[:-1], seq_lh[1:]):
    N_lh[pos10[a], pos10[b]] += 1
P_lh = (N_lh + ALPHA) / (N_lh + ALPHA).sum(axis=1, keepdims=True)

ev_lh = np.linalg.eigvals(P_lh)
ev_lh = ev_lh[np.argsort(-np.abs(ev_lh))]
l2_lh = abs(ev_lh[1])

null_lh = np.empty(2000)
arr_lh = np.array(seq_lh, dtype=object)
for b in range(2000):
    pl = list(rng.permutation(arr_lh))
    Nb = np.zeros((M10, M10))
    for a, c in zip(pl[:-1], pl[1:]):
        Nb[pos10[a], pos10[c]] += 1
    Pb = (Nb + ALPHA) / (Nb + ALPHA).sum(axis=1, keepdims=True)
    evb = np.linalg.eigvals(Pb)
    null_lh[b] = abs(evb[np.argsort(-np.abs(evb))][1])
p_lh = (np.sum(null_lh >= l2_lh) + 1) / 2001

print("\n--- LAGGARD, 10 sectors, 1989-2026 ---")
print(f"  weekly transitions : {len(seq_lh)-1}")
print(f"  |lambda_2|         : {l2_lh:.5f}")
print(f"  max imag part      : {np.max(np.abs(ev_lh.imag)):.5f}")
print(f"  null mean / 97.5%  : {null_lh.mean():.5f} / {np.percentile(null_lh,97.5):.5f}")
print(f"  p-value            : {p_lh:.4f}")
print(f"  verdict            : {'SIGNAL' if l2_lh > np.percentile(null_lh,97.5) else 'NOISE (no memory)'}")

long-history panel: (9260, 10) | 1989-09-11 -> 2026-06-17
weekly states: 1919

--- LAGGARD, 10 sectors, 1989-2026 ---
  weekly transitions : 1918
  |lambda_2|         : 0.09914
  max imag part      : 0.03847
  null mean / 97.5%  : 0.06834 / 0.09360
  p-value            : 0.0095
  verdict            : SIGNAL


In [60]:
print("=" * 64)
print("LONG-HISTORY (10 sectors, 1989-2026) — SIGNAL STRESS TEST")
print("=" * 64)

print("\n[1] lambda_2 decomposition: real or complex?")
print("  top-4 eigenvalues of P_lh:")
for v in ev_lh[:4]:
    tag = " <- lambda_2" if abs(abs(v) - l2_lh) < 1e-9 else ""
    print(f"    Re={v.real:+.5f}  Im={v.imag:+.5f}  mod={abs(v):.5f}{tag}")
diag_lh = np.mean(np.diag(P_lh))
print(f"  mean diagonal P[i,i]: {diag_lh:.4f}  | uniform 1/10 = {1/M10:.4f}")
print("  => if lambda_2 real and diag>uniform: weak persistence, not a cycle")

print("\n[2] BLOCK BOOTSTRAP null (preserves short-run autocorrelation)")
def block_null_l2(seq_list, posmap, msz, blocklen=4, nb=2000):
    arr = np.array(seq_list, dtype=object); nn = len(arr)
    nblk = int(np.ceil(nn / blocklen))
    blocks = [arr[i*blocklen:(i+1)*blocklen] for i in range(nblk)]
    out = np.empty(nb)
    for b in range(nb):
        order = rng.permutation(len(blocks))
        pl = list(np.concatenate([blocks[i] for i in order]))[:nn]
        Nb = np.zeros((msz, msz))
        for a, c in zip(pl[:-1], pl[1:]):
            Nb[posmap[a], posmap[c]] += 1
        Pb = (Nb + ALPHA) / (Nb + ALPHA).sum(axis=1, keepdims=True)
        evb = np.linalg.eigvals(Pb)
        out[b] = abs(evb[np.argsort(-np.abs(evb))][1])
    return out

blk = block_null_l2(seq_lh, pos10, M10)
p_blk = (np.sum(blk >= l2_lh) + 1) / (len(blk) + 1)
print(f"  |lambda_2|={l2_lh:.5f}")
print(f"  simple-shuffle null 97.5% : {np.percentile(null_lh,97.5):.5f}  (p={p_lh:.4f})")
print(f"  block-bootstrap null 97.5%: {np.percentile(blk,97.5):.5f}  (p={p_blk:.4f})")
print(f"  verdict: {'SIGNAL survives block test' if p_blk < 0.05 else 'GONE under block null (was autocorr artifact)'}")

print("\n[3] MARKET-DEMEANED daily lead-lag (10 sectors)")
Rf = rets_lh.astype("float64")
D = Rf.sub(Rf.mean(axis=1), axis=0)
X = D.iloc[:-1].values; Y = D.iloc[1:].values
T = X.shape[0]
Xc = X - X.mean(0); Yc = Y - Y.mean(0)
C = (Xc.T @ Yc) / (np.sqrt((Xc**2).sum(0))[:,None] * np.sqrt((Yc**2).sum(0))[None,:])
fro = np.sqrt((C**2).sum())
nullf = np.empty(2000)
for b in range(2000):
    sh = rng.integers(1, T)
    Yp = np.roll(Y, sh, axis=0); Ypc = Yp - Yp.mean(0)
    Cb = (Xc.T @ Ypc) / (np.sqrt((Xc**2).sum(0))[:,None] * np.sqrt((Ypc**2).sum(0))[None,:])
    nullf[b] = np.sqrt((Cb**2).sum())
p_ll = (np.sum(nullf >= fro) + 1) / 2001
off = C[~np.eye(M10, dtype=bool)]
print(f"  T={T}  off-diag mean|c|={np.abs(off).mean():.4f}  frac_negative={(off<0).mean():.2f}")
print(f"  ||C||_F={fro:.4f}  cyclic-shift null 97.5%={np.percentile(nullf,97.5):.4f}  p={p_ll:.4f}")
print(f"  verdict: {'STRUCTURE survives' if p_ll < 0.05 else 'GONE (common-factor / no usable lead-lag)'}")

print("\n[4] PLACEBO LAG (should decay to ~0 if real 1-step structure)")
for lag in [1, 5, 20, 60]:
    Xl = D.values[:-lag]; Yl = D.values[lag:]
    Xlc = Xl - Xl.mean(0); Ylc = Yl - Yl.mean(0)
    Cl = (Xlc.T @ Ylc) / (np.sqrt((Xlc**2).sum(0))[:,None] * np.sqrt((Ylc**2).sum(0))[None,:])
    print(f"  lag={lag:>2}: ||C||_F={np.sqrt((Cl**2).sum()):.4f}")

LONG-HISTORY (10 sectors, 1989-2026) — SIGNAL STRESS TEST

[1] lambda_2 decomposition: real or complex?
  top-4 eigenvalues of P_lh:
    Re=+1.00000  Im=+0.00000  mod=1.00000
    Re=+0.09914  Im=+0.00000  mod=0.09914 <- lambda_2
    Re=-0.06150  Im=+0.00000  mod=0.06150
    Re=+0.05127  Im=+0.00000  mod=0.05127
  mean diagonal P[i,i]: 0.1071  | uniform 1/10 = 0.1000
  => if lambda_2 real and diag>uniform: weak persistence, not a cycle

[2] BLOCK BOOTSTRAP null (preserves short-run autocorrelation)
  |lambda_2|=0.09914
  simple-shuffle null 97.5% : 0.09360  (p=0.0095)
  block-bootstrap null 97.5%: 0.09041  (p=0.0065)
  verdict: SIGNAL survives block test

[3] MARKET-DEMEANED daily lead-lag (10 sectors)
  T=9258  off-diag mean|c|=0.0248  frac_negative=0.56
  ||C||_F=0.3219  cyclic-shift null 97.5%=0.1332  p=0.0005
  verdict: STRUCTURE survives

[4] PLACEBO LAG (should decay to ~0 if real 1-step structure)
  lag= 1: ||C||_F=0.3219
  lag= 5: ||C||_F=0.1173
  lag=20: ||C||_F=0.1398
  lag=60

## 3. State Construction & Transition Matrix

Collapse each week's 11-dimensional return vector into a single categorical state:
the worst-performing sector that week (the "laggard"). This is the step that makes
a Markov chain well-defined — the market occupies exactly one state per period.
From the resulting state sequence, count transitions, estimate the maximum-likelihood
transition matrix P, and apply Laplace smoothing to remove zero cells.

In [62]:
weekly_ret = rets.resample("W-FRI").sum()
weekly_ret = weekly_ret[rets.resample("W-FRI").size() > 0]

state = weekly_ret.idxmin(axis=1)
state.name = "laggard"
seq = state.tolist()

print("weeks (states):", len(state))
print("trading-days-per-week:",
      rets.resample("W-FRI").size().value_counts().sort_index().to_dict())
print("ties (>1 sector at min):",
      int((weekly_ret.eq(weekly_ret.min(axis=1), axis=0)).sum(axis=1).gt(1).sum()))
print("\nlaggard frequency by sector:")
print(state.value_counts().reindex(SECTORS).to_string())

weeks (states): 509
trading-days-per-week: {3: 1, 4: 94, 5: 414}
ties (>1 sector at min): 0

laggard frequency by sector:
laggard
Energy               127
Materials             27
Industrials           13
ConsDiscretionary     40
ConsStaples           34
InfoTech              43
HealthCare            34
Financials            34
CommServices          53
Utilities             61
RealEstate            43
